# Building End-to-End ML Pipelines
**Level:** Intermediate | **Interactive Machine Learning Curriculum**

> Constructing modular production ML pipelines with Scikit-learn: ColumnTransformer, Pipeline, FeatureUnion, custom BaseEstimator transformers, and serialization.

---

### 📖 Conceptual Foundations

#### What is an ML Pipeline?

A **Pipeline** chains multiple data transformation steps sequentially, terminating in a final estimator (model). It provides a unified API where calling **`pipeline.fit(X, y)`** executes all transformers and fits the model, and **`pipeline.predict(X_new)`** transforms raw inference data through all steps automatically.


#### Why Pipelines Are Mandatory in Production

1. **Zero Data Leakage:** Guarantees that scalers, encoders, and imputers learn statistics **strictly from training folds** during cross-validation, applying only `transform()` to validation/test sets.
2. **Atomic Deployment:** The entire preprocessing recipe and model weights serialize as a **single atomic file (`model.joblib`)**, eliminating manual glue code at serving time.
3. **Unified Hyperparameter Tuning:** Allows simultaneous optimization of preprocessing parameters and model hyperparameters (e.g. tuning PCA component count and classifier regularizer $C$ inside one `GridSearchCV`).


#### Pipeline Architectural Building Blocks

| Class | Role | Input / Output Behavior |
|---|---|---|
| **`Pipeline`** | Chained sequential execution | Output of step $i$ becomes input to step $i+1$; final step must be an Estimator |
| **`ColumnTransformer`** | Parallel feature-specific routing | Applies specific transformers to specified subsets of columns and concatenates results horizontally |
| **`FeatureUnion`** | Parallel transformation | Applies multiple transformers to the entire dataset and concatenates extracted feature matrices side-by-side |
| **Custom Transformer** | Domain-specific logic | Inherits from `BaseEstimator` and `TransformerMixin`; implements `fit()` and `transform()` |


#### The Estimator vs. Transformer Contract

- **Transformer:** Must implement `fit(X, y)` and `transform(X)`. Returns transformed feature matrix $X'$.
- **Estimator:** Must implement `fit(X, y)` and `predict(X)` (or `predict_proba`). Cannot appear in the middle of a pipeline; only as the final step!


#### Senior Engineer Rule

Never write raw, loose preprocessing code in notebooks that must be rewritten in production services. Build a Scikit-learn `Pipeline` on Day 1: it runs identically in notebooks, cross-validation, and production microservice endpoints.



In [ ]:
# %matplotlib inline
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Custom domain transformer: calculates ratios
class DebtRatioExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        # Assumes X[:, 0] is debt, X[:, 1] is income
        debt = X[:, 0]
        income = np.maximum(X[:, 1], 1.0)  # avoid division by zero
        ratio = (debt / income).reshape(-1, 1)
        return np.hstack([X, ratio])

# Sample data
df = pd.DataFrame({
    'debt': [5000, 15000, 2000, np.nan, 8000, 12000],
    'income': [50000, 45000, 30000, 80000, 60000, 35000],
    'state': ['NY', 'CA', 'TX', 'NY', 'CA', 'TX'],
    'default': [0, 1, 0, 0, 0, 1]
})

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('ratio', DebtRatioExtractor()),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, ['debt', 'income']),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), ['state'])
])

full_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(n_estimators=10, random_state=42))
])

full_pipeline.fit(df.drop('default', axis=1), df['default'])
print('Pipeline successfully fitted and verified!')
print('Final prediction for row 0:', full_pipeline.predict(df.iloc[[0]].drop('default', axis=1))[0])


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

